# 07 — Demo

Interactive prediction on hand-written campaign snippets. Each example is run
through all three trained models (TF-IDF + LR, DistilBERT, retrieval-enhanced)
and the results are shown side-by-side.


In [1]:
# Path-setup boilerplate so the notebook can import src.*
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


project root: /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)


In [2]:
import joblib, numpy as np, pandas as pd, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src import config
from src import retrieval_pipeline as rp
from src.models import retrieval_enhanced_classifier as rec


## 1. Load the three artefacts

In [3]:
tfidf_lr = joblib.load(config.MODELS_DIR / 'tfidf_lr.joblib')
tok = AutoTokenizer.from_pretrained(str(config.MODELS_DIR / 'distilbert'))
bert = AutoModelForSequenceClassification.from_pretrained(str(config.MODELS_DIR / 'distilbert'))
bert.eval()
embs = np.load(config.EMBEDDINGS_NPY)
df = pd.read_csv(config.PROCESSED_CSV)
print('all three models loaded')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

all three models loaded


## 2. Compose example campaigns

In [4]:
examples = [
    # 1. Strong urgency + sympathy
    'PLEASE HELP IMMEDIATELY — my mother only has DAYS LEFT. She is alone, '
    'helpless, and we have nothing. Every second counts before it is too late.',
    # 2. Neutral medical bills request
    'My father had heart surgery last month. We are raising funds to help cover '
    'the medical expenses and recovery costs. Any contribution is appreciated.',
    # 3. Memorial / emotional but factual
    'My brother passed away on Sunday after a long illness. We are raising '
    'money to cover the funeral and to support his children with school costs.',
    # 4. Animals
    'Our rescue dog Bella needs an emergency operation. Please consider '
    'donating to her surgery fund if you can spare anything.',
]
for i, e in enumerate(examples, 1):
    print(f'{i}.', e[:90], '...\n')


1. PLEASE HELP IMMEDIATELY — my mother only has DAYS LEFT. She is alone, helpless, and we hav ...

2. My father had heart surgery last month. We are raising funds to help cover the medical exp ...

3. My brother passed away on Sunday after a long illness. We are raising money to cover the f ...

4. Our rescue dog Bella needs an emergency operation. Please consider donating to her surgery ...



## 3. Run all three models

In [5]:
def predict_all(texts):
    out = pd.DataFrame({'text': [t[:50] + '...' for t in texts]})
    # TF-IDF + LR
    out['lr_pred'] = [config.ID2LABEL[i] for i in tfidf_lr.predict(texts)]
    out['lr_proba_man'] = tfidf_lr.predict_proba(texts)[:,1].round(3)
    # DistilBERT
    enc = tok(texts, truncation=True, padding=True,
              max_length=config.TRANSFORMER_MAX_LEN, return_tensors='pt')
    with torch.no_grad():
        logits = bert(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    out['bert_pred'] = [config.ID2LABEL[i] for i in probs.argmax(axis=1)]
    out['bert_proba_man'] = probs[:,1].round(3)
    # Retrieval-enhanced
    q = rp.embed(texts, show_progress=False)
    index = rp.build_index(embs)
    sims, neigh = rp.topk_neighbours(index, q, k=config.RETRIEVAL_TOP_K)
    train_lab = df['binary_label'].map(config.LABEL2ID).values
    rpred, rproba = rec.predict_combined(probs, neigh, sims, train_lab, alpha=0.7)
    out['retr_pred'] = [config.ID2LABEL[i] for i in rpred]
    out['retr_proba_man'] = rproba[:,1].round(3)
    return out

predict_all(examples)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,text,lr_pred,lr_proba_man,bert_pred,bert_proba_man,retr_pred,retr_proba_man
0,PLEASE HELP IMMEDIATELY — my mother only has D...,manipulative,0.979,manipulative,0.527,non_manipulative,0.427
1,My father had heart surgery last month. We are...,non_manipulative,0.191,non_manipulative,0.096,non_manipulative,0.067
2,My brother passed away on Sunday after a long ...,manipulative,0.950,manipulative,0.977,manipulative,0.984
3,Our rescue dog Bella needs an emergency operat...,non_manipulative,0.156,non_manipulative,0.191,non_manipulative,0.193


**What to look for.** All three models should label example 1 (urgency +
sympathy) as `manipulative` with high probability and example 2 (neutral
medical-bills) as `non_manipulative`. Examples 3 and 4 are the interesting
edge cases — genuinely emotional but not necessarily manipulative — and the
three models often disagree, illustrating the limits of the weak-label
training signal.


**End of demo.** For a fuller analysis of where the models agree and disagree,
see notebook `05_comparison.ipynb` and §5.4 (error analysis) in the final report.
